
# N11: The Grandmaster Pipeline (Ultimate Seed-Blended Pseudo-Labeling)
**Objective:** Execute the absolute final, peak Kaggle generalization pipeline. Based on N10's ablation victory (Seed-Blending) and our empirical ceiling mapping, this pipeline combines Architectural Diversity (CatBoost, XGBoost, LightGBM), Stochastic Regularization (3 seeds per model), and High-Confidence Pseudo-Labeling (>99%) to mathematically squeeze out the absolute maximum un-leaked score.


In [ ]:

import pandas as pd
import numpy as np
import warnings
import torch
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')

ID_COL, TARGET_COL = "id", "health_condition"
CLASSES = ("at-risk", "fit", "unhealthy")
N_FOLDS = 5
SEEDS = [2026, 42, 999]
GPU_ENABLED = torch.cuda.is_available()
print(f"GPU Available: {GPU_ENABLED}")


In [ ]:

# 1. Load Data
train_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/test.csv')

for df in [train_df, test_df]:
    df['sleep_bin'] = pd.qcut(df['sleep_duration'], q=5, duplicates='drop').astype(str)
    df['stress_sleep_interact'] = df['stress_level'].astype(str) + '_' + df['sleep_bin']

cat_cols = ['stress_level', 'physical_activity_level', 'diet_type', 'gender', 'smoking_alcohol', 'sleep_quality', 'stress_sleep_interact']
num_cols = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']

le_target = LabelEncoder()
y_train = le_target.fit_transform(train_df[TARGET_COL])
X_train_raw = train_df[cat_cols + num_cols].copy()
X_test_raw = test_df[cat_cols + num_cols].copy()

for col in cat_cols:
    X_train_raw[col] = X_train_raw[col].fillna('Missing').astype(str)
    X_test_raw[col] = X_test_raw[col].fillna('Missing').astype(str)

num_imputer = SimpleImputer(strategy='median')
X_train_num_raw = num_imputer.fit_transform(train_df[num_cols])
X_test_num_raw = num_imputer.transform(test_df[num_cols])


In [ ]:

# 2. Phase 1: Base Target Encoding & Model Definition
def get_te_matrices(tr_i, va_i):
    X_tr_cat, X_va_cat = X_train_raw[cat_cols].iloc[tr_i], X_train_raw[cat_cols].iloc[va_i]
    X_tr_num, X_va_num = X_train_num_raw[tr_i], X_train_num_raw[va_i]
    y_tr = y_train[tr_i]
    
    fold_te_tr, fold_te_val, fold_te_test = [], [], []
    for col in cat_cols:
        crosstab = pd.crosstab(X_tr_cat[col], y_tr, normalize='index')
        mapping = crosstab.to_dict('index')
        mean_mapping = crosstab.mean().to_dict()
        
        tr_mapped = X_tr_cat[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
        fold_te_tr.append(pd.DataFrame(tr_mapped.tolist()).values)
        va_mapped = X_va_cat[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
        fold_te_val.append(pd.DataFrame(va_mapped.tolist()).values)
        te_mapped = X_test_raw[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
        fold_te_test.append(pd.DataFrame(te_mapped.tolist()).values)
        
    X_tr_full = np.hstack([X_tr_num, np.hstack(fold_te_tr)])
    X_va_full = np.hstack([X_va_num, np.hstack(fold_te_val)])
    X_te_full = np.hstack([X_test_num_raw, np.hstack(fold_te_test)])
    return X_tr_full, X_va_full, X_te_full

def train_ensemble(X_tr, y_tr, X_va, X_te, weights, sample_weights):
    fold_preds_va = np.zeros((len(X_va), 3))
    fold_preds_te = np.zeros((len(X_te), 3))
    n_models = 3 * len(SEEDS)
    
    cb_task = "GPU" if GPU_ENABLED else "CPU"
    xgb_tree = "hist"
    xgb_device = "cuda" if GPU_ENABLED else "cpu"
    
    for s in SEEDS:
        # 1. CatBoost
        m_cb = CatBoostClassifier(iterations=800, learning_rate=0.03, depth=6, class_weights=weights, random_seed=s, verbose=0, task_type=cb_task)
        m_cb.fit(X_tr, y_tr)
        fold_preds_va += m_cb.predict_proba(X_va) / n_models
        fold_preds_te += m_cb.predict_proba(X_te) / n_models
        
        # 2. XGBoost
        m_xgb = XGBClassifier(n_estimators=800, learning_rate=0.03, max_depth=6, random_state=s, n_jobs=-1, objective='multi:softprob', tree_method=xgb_tree, device=xgb_device)
        m_xgb.fit(X_tr, y_tr, sample_weight=sample_weights, verbose=0)
        fold_preds_va += m_xgb.predict_proba(X_va) / n_models
        fold_preds_te += m_xgb.predict_proba(X_te) / n_models
        
        # 3. LightGBM
        m_lgb = LGBMClassifier(n_estimators=800, learning_rate=0.03, num_leaves=31, class_weight='balanced', random_state=s, n_jobs=-1, verbose=-1)
        m_lgb.fit(X_tr, y_tr)
        fold_preds_va += m_lgb.predict_proba(X_va) / n_models
        fold_preds_te += m_lgb.predict_proba(X_te) / n_models
        
    return fold_preds_va, fold_preds_te


In [ ]:

# 3. Phase 2: Base Ensemble Training & Pseudo-Label Generation
print("--- Phase 2: Training Base 9-Model Seed Blend ---")
base_tst_probs = np.zeros((len(test_df), 3))
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=2026)

for fold, (tr_i, va_i) in enumerate(skf.split(X_train_raw, y_train)):
    X_tr_full, X_va_full, X_te_full = get_te_matrices(tr_i, va_i)
    
    class_counts = np.bincount(y_train[tr_i])
    weights = [len(y_train[tr_i]) / (len(class_counts) * c) for c in class_counts]
    from sklearn.utils.class_weight import compute_sample_weight
    sample_weights = compute_sample_weight('balanced', y_train[tr_i])
    
    _, fold_te_preds = train_ensemble(X_tr_full, y_train[tr_i], X_va_full, X_te_full, weights, sample_weights)
    base_tst_probs += fold_te_preds / N_FOLDS
    print(f"Base Fold {fold+1} Complete.")

CONF_THRESH = 0.99
max_probs = np.max(base_tst_probs, axis=1)
pseudo_idx = np.where(max_probs >= CONF_THRESH)[0]
pseudo_labels = np.argmax(base_tst_probs[pseudo_idx], axis=1)

print(f"\nExtracted Pseudo-Labels: {len(pseudo_idx)} rows (>99% confidence)")


In [ ]:

# 4. Phase 3: Augmented Retraining (The Final Pass)
print("\n--- Phase 3: Augmented Retraining ---")
aug_tst_probs = np.zeros((len(test_df), 3))

# Append Pseudo-Labels to training set
aug_X_train_raw = pd.concat([X_train_raw, X_test_raw.iloc[pseudo_idx]]).reset_index(drop=True)
aug_X_train_num = np.vstack([X_train_num_raw, X_test_num_raw[pseudo_idx]])
aug_y_train = np.concatenate([y_train, pseudo_labels])

skf_aug = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

for fold, (tr_i, va_i) in enumerate(skf_aug.split(aug_X_train_raw, aug_y_train)):
    # Re-calculate Target Encoding dynamically on Augmented folds
    X_tr_cat, X_va_cat = aug_X_train_raw[cat_cols].iloc[tr_i], aug_X_train_raw[cat_cols].iloc[va_i]
    X_tr_num, X_va_num = aug_X_train_num[tr_i], aug_X_train_num[va_i]
    y_tr = aug_y_train[tr_i]
    
    fold_te_tr, fold_te_val, fold_te_test = [], [], []
    for col in cat_cols:
        crosstab = pd.crosstab(X_tr_cat[col], y_tr, normalize='index')
        mapping = crosstab.to_dict('index')
        mean_mapping = crosstab.mean().to_dict()
        
        tr_mapped = X_tr_cat[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
        fold_te_tr.append(pd.DataFrame(tr_mapped.tolist()).values)
        va_mapped = X_va_cat[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
        fold_te_val.append(pd.DataFrame(va_mapped.tolist()).values)
        te_mapped = X_test_raw[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
        fold_te_test.append(pd.DataFrame(te_mapped.tolist()).values)
        
    X_tr_full = np.hstack([X_tr_num, np.hstack(fold_te_tr)])
    X_va_full = np.hstack([X_va_num, np.hstack(fold_te_val)])
    X_te_full = np.hstack([X_test_num_raw, np.hstack(fold_te_test)])
    
    class_counts = np.bincount(y_tr)
    weights = [len(y_tr) / (len(class_counts) * c) for c in class_counts]
    from sklearn.utils.class_weight import compute_sample_weight
    sample_weights = compute_sample_weight('balanced', y_tr)
    
    _, fold_te_preds = train_ensemble(X_tr_full, y_tr, X_va_full, X_te_full, weights, sample_weights)
    aug_tst_probs += fold_te_preds / N_FOLDS
    print(f"Augmented Fold {fold+1} Complete.")


In [ ]:

# 5. Output Final Grandmaster Predictions
final_preds = aug_tst_probs.argmax(axis=1)

sub_df = pd.DataFrame({
    ID_COL: test_df[ID_COL].astype("int64"),
    TARGET_COL: [CLASSES[i] for i in final_preds]
})
sub_df.to_csv("submission.csv", index=False)
print("\nExported Grandmaster Pipeline submission.csv")
